# Notebook 10 — Significance test for the cross-transfer edge

Tests whether MAML-transfer really beats conventional (static) pre-transfer, across
independent training seeds, not just support draws. For each seed we meta-train MAML on the
source plant, train a static AE on the source plant, and evaluate both on the target plant
under identical support draws. We then run a paired test on the per-seed AUROC differences
and report the mean difference with a 95% confidence interval. The direction that carries the
claim is WADI to SWaT.

**Design.** The paired unit is the training seed. Both models are retrained per seed, so the
test isolates the method, not the support sampling. Meta-training uses early stopping on the
meta-validation loss, which keeps runtime bounded on low-diversity sources.

### ▶ Run: attach BOTH `swat-maml-data` and `wadi-maml-data`. GPU. Save & Run All (Commit).
Per-seed results are saved and auto-resumed, so an interrupted run continues from the next seed.


## 1 — Imports, data, PCA projection, resume store

In [1]:
import os, json, pickle, copy, time, warnings
import numpy as np, torch, torch.nn as nn
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score
from scipy import stats
warnings.filterwarnings('ignore')
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print("Device:",DEVICE)
OUT="/kaggle/working"; os.makedirs(f"{OUT}/results",exist_ok=True)
RESULTS_PATH=f"{OUT}/results/sig_results.json"

def _find(name):
    for r,_,f in os.walk("/kaggle/input"):
        if name in f: return os.path.join(r,name)
    return None
def load_plant(p):
    return (np.load(_find(f"{p}_normal.npy")).astype(np.float32),
            np.load(_find(f"{p}_attack.npy")).astype(np.float32),
            np.load(_find(f"{p}_attack_labels.npy")),
            pickle.load(open(_find(f"{p}_tasks.pkl"),"rb")),
            json.load(open(_find(f"{p}_task_splits.json"))))
DATA={p:load_plant(p) for p in ["swat","wadi"]}

D=32; W=30; S=10
def make_windows(a): return np.array([a[i:i+W] for i in range(0,len(a)-W+1,S)],dtype=np.float32)
PROJ={}
for p,(Xn,Xa,ya,tasks,splits) in DATA.items():
    pca=PCA(n_components=D,random_state=42).fit(Xn)
    tn=pca.transform(Xn); lo=tn.min(0); hi=tn.max(0); rg=np.where(hi-lo>1e-8,hi-lo,1.0)
    pr=lambda x2d,pca=pca,lo=lo,rg=rg: np.clip((pca.transform(x2d)-lo)/rg,0,1).astype(np.float32)
    ptasks={}
    for k,v in tasks.items():
        w=v["normal_windows"]; n,T,F=w.shape
        ptasks[k]=pr(w.reshape(n*T,F)).reshape(n,T,D).astype(np.float32)
    PROJ[p]={"attack":pr(Xa),"labels":ya,"tasks":ptasks,"splits":splits,
             "normal_windows":make_windows(pr(Xn))}
    print(f"{p}: projected to {D}d, normal windows {PROJ[p]['normal_windows'].shape}")

# resume: load any prior per-seed results from working or attached inputs
def load_prior():
    paths=[RESULTS_PATH]+[os.path.join(r,'sig_results.json') for r,_,f in os.walk('/kaggle/input') if 'sig_results.json' in f]
    merged={}
    for p in paths:
        if os.path.exists(p):
            try:
                d=json.load(open(p))
                for k,v in d.items(): merged.setdefault(k,{}).update(v)
            except: pass
    return merged
RESULTS=load_prior()
print("resumed directions/seeds:", {k:sorted(v.keys()) for k,v in RESULTS.items()})


Device: cuda
swat: projected to 32d, normal windows (4948, 30, 32)
wadi: projected to 32d, normal windows (7843, 30, 32)
resumed directions/seeds: {}


## 2 — Model, corrected FOMAML core, static trainer, transfer eval

In [2]:
class Enc(nn.Module):
    def __init__(s,f,h1=64,h2=32,z=16):
        super().__init__(); s.l1=nn.LSTM(f,h1,batch_first=True); s.l2=nn.LSTM(h1,h2,batch_first=True); s.fc=nn.Linear(h2,z)
    def forward(s,x): o,_=s.l1(x); _,(h,_)=s.l2(o); return s.fc(h.squeeze(0))
class Dec(nn.Module):
    def __init__(s,z=16,h1=32,h2=64,f=32,seq=30):
        super().__init__(); s.seq=seq; s.l1=nn.LSTM(z,h1,batch_first=True); s.l2=nn.LSTM(h1,h2,batch_first=True); s.fc=nn.Linear(h2,f)
    def forward(s,z): r=z.unsqueeze(1).repeat(1,s.seq,1); o,_=s.l1(r); o,_=s.l2(o); return s.fc(o)
class LSTMAE(nn.Module):
    def __init__(s,f=32): super().__init__(); s.encoder=Enc(f); s.decoder=Dec(f=f)
    def forward(s,x): return s.decoder(s.encoder(x))
    def recon(s,x): xh=s.forward(x); return torch.mean((x-xh)**2,dim=(1,2))
criterion=nn.MSELoss()

def sample_ep(w,ss,qs,rng):
    idx=rng.permutation(len(w)); need=ss+qs
    if len(w)<need: a=w[rng.choice(len(w),ss,replace=True)]; b=w[rng.choice(len(w),qs,replace=True)]
    else: a=w[idx[:ss]]; b=w[idx[ss:need]]
    return torch.tensor(a,dtype=torch.float32).to(DEVICE), torch.tensor(b,dtype=torch.float32).to(DEVICE)
def inner(model,support,lr=0.01,steps=10):
    L=copy.deepcopy(model); L.train(); o=torch.optim.SGD(L.parameters(),lr=lr)
    for _ in range(steps): o.zero_grad(); l=criterion(L(support),support); l.backward(); o.step()
    return L
def outer(model,opt,batch,rng,steps=10,ss=20,qs=20,train=True):
    acc=[None]*len(list(model.parameters())); ml=0.0
    for w in batch:
        sup,qry=sample_ep(w,ss,qs,rng); Ln=inner(model,sup,0.01,steps); ql=criterion(Ln(qry),qry)
        if train:
            g=torch.autograd.grad(ql,Ln.parameters()); acc=[gi.detach() if a is None else a+gi.detach() for a,gi in zip(acc,g)]
        ml+=ql.item()
    ml/=len(batch)
    if train:
        opt.zero_grad()
        for p,a in zip(model.parameters(),acc): p.grad=a/len(batch)
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
    return ml

def meta_train(src,seed,max_outer=5000,val_every=250,patience=6):
    torch.manual_seed(seed); np.random.seed(seed); rng=np.random.RandomState(seed)
    model=LSTMAE(D).to(DEVICE); opt=torch.optim.Adam(model.parameters(),lr=1e-3)
    regs=[PROJ[src]["tasks"][k] for k in PROJ[src]["splits"]["meta_train"]]
    vals=[PROJ[src]["tasks"][k] for k in PROJ[src]["splits"]["meta_val"]]
    best=1e9; best_state={k:v.clone() for k,v in model.state_dict().items()}; bad=0
    for step in range(1,max_outer+1):
        b=[regs[i] for i in rng.choice(len(regs),min(4,len(regs)),replace=False)]
        model.train(); outer(model,opt,b,rng,steps=10,ss=20,qs=20,train=True)
        if step%val_every==0:
            vl=float(np.mean([outer(model,opt,[v],rng,steps=10,ss=20,qs=20,train=False) for v in vals]))
            if vl<best-1e-6: best=vl; best_state={k:v.clone() for k,v in model.state_dict().items()}; bad=0
            else:
                bad+=1
                if bad>=patience: break
    model.load_state_dict(best_state); return model

def train_static(src,seed,max_ep=120,patience=12):
    torch.manual_seed(seed); np.random.seed(seed)
    m=LSTMAE(D).to(DEVICE); o=torch.optim.Adam(m.parameters(),lr=1e-3)
    data=torch.tensor(PROJ[src]["normal_windows"]); p=torch.randperm(len(data)); cut=int(0.9*len(data))
    tr=data[p[:cut]]; va=data[p[cut:]].to(DEVICE); best=1e9; bad=0; bs=None
    for ep in range(max_ep):
        m.train(); pi=torch.randperm(len(tr))
        for st in range(0,len(tr),128):
            bb=tr[pi[st:st+128]].to(DEVICE); o.zero_grad(); l=criterion(m(bb),bb); l.backward(); o.step()
        m.eval()
        with torch.no_grad(): vl=criterion(m(va),va).item()
        if vl<best-1e-6: best=vl; bad=0; bs={k:v.clone() for k,v in m.state_dict().items()}
        else:
            bad+=1
            if bad>=patience: break
    m.load_state_dict(bs); return m

def transfer_auc(model,tgt,support_size=50,support_seeds=(0,1,2)):
    Xa=PROJ[tgt]["attack"]; ya=PROJ[tgt]["labels"]
    idx=[(i,i+W) for i in range(0,len(Xa)-W+1,S)]
    Wa=torch.tensor(np.array([Xa[a:b] for a,b in idx],dtype=np.float32)).to(DEVICE)
    yw=np.array([int(ya[a:b].any()) for a,b in idx])
    nw=PROJ[tgt]["normal_windows"]; aucs=[]
    for ss_seed in support_seeds:
        r=np.random.RandomState(ss_seed)
        sup=torch.tensor(nw[r.choice(len(nw),support_size,replace=False)],dtype=torch.float32).to(DEVICE)
        Ln=inner(model,sup,0.01,10); Ln.eval()
        with torch.no_grad(): sc=Ln.recon(Wa).cpu().numpy()
        aucs.append(roc_auc_score(yw,sc))
    return float(np.mean(aucs))
print("ready")


ready


## 3 — Seeded sweep (per-seed resume). WADI to SWaT is the claim direction.

In [3]:
SEEDS=list(range(6))          # 10 independent training seeds
def run_direction(src,tgt,tag):
    RESULTS.setdefault(tag,{})
    for seed in SEEDS:
        if str(seed) in RESULTS[tag]:
            continue
        t0=time.time()
        mm=meta_train(src,seed); ms=train_static(src,seed)
        maml_auc=transfer_auc(mm,tgt); static_auc=transfer_auc(ms,tgt)
        RESULTS[tag][str(seed)]={"maml":round(maml_auc,4),"static":round(static_auc,4)}
        json.dump(RESULTS,open(RESULTS_PATH,"w"),indent=2)   # persist after every seed
        print(f"[{tag}] seed {seed}: MAML {maml_auc:.4f}  Static {static_auc:.4f}  d {maml_auc-static_auc:+.4f}  ({time.time()-t0:.0f}s)")
    print(f"[{tag}] complete")

run_direction("wadi","swat","WADI_to_SWaT")   # claim direction first


[WADI_to_SWaT] seed 0: MAML 0.7917  Static 0.7962  d -0.0045  (1317s)
[WADI_to_SWaT] seed 1: MAML 0.7820  Static 0.8149  d -0.0329  (669s)
[WADI_to_SWaT] seed 2: MAML 0.8062  Static 0.7959  d +0.0102  (1312s)
[WADI_to_SWaT] seed 3: MAML 0.7981  Static 0.8039  d -0.0058  (1288s)
[WADI_to_SWaT] seed 4: MAML 0.7912  Static 0.7999  d -0.0087  (1300s)
[WADI_to_SWaT] seed 5: MAML 0.8082  Static 0.7706  d +0.0376  (1302s)
[WADI_to_SWaT] complete


In [4]:
run_direction("swat","wadi","SWaT_to_WADI")   # reverse direction, for symmetry (expected null)


[SWaT_to_WADI] seed 0: MAML 0.5823  Static 0.5883  d -0.0059  (1160s)
[SWaT_to_WADI] seed 1: MAML 0.5467  Static 0.5998  d -0.0530  (1016s)
[SWaT_to_WADI] seed 2: MAML 0.6359  Static 0.5966  d +0.0392  (778s)
[SWaT_to_WADI] seed 3: MAML 0.5473  Static 0.5912  d -0.0439  (1277s)
[SWaT_to_WADI] seed 4: MAML 0.5377  Static 0.5945  d -0.0568  (708s)
[SWaT_to_WADI] seed 5: MAML 0.5955  Static 0.6017  d -0.0062  (894s)
[SWaT_to_WADI] complete


## 4 — Paired statistics and report

In [5]:
def paired_report(tag):
    r=RESULTS.get(tag,{})
    seeds=sorted(r.keys(),key=int)
    m=np.array([r[s]["maml"] for s in seeds]); st=np.array([r[s]["static"] for s in seeds])
    d=m-st; n=len(d)
    mean_d=d.mean(); sd=d.std(ddof=1); sem=sd/np.sqrt(n)
    tcrit=stats.t.ppf(0.975,n-1); ci=(mean_d-tcrit*sem, mean_d+tcrit*sem)
    t_p=stats.ttest_rel(m,st,alternative='greater').pvalue
    try: w_p=stats.wilcoxon(m,st,alternative='greater').pvalue
    except Exception: w_p=float('nan')
    print("="*60); print(f"DIRECTION: {tag}   (n={n} training seeds)"); print("="*60)
    print(f"{'seed':>4} | {'MAML':>7} | {'Static':>7} | {'diff':>8}")
    for s in seeds: print(f"{s:>4} | {r[s]['maml']:>7.4f} | {r[s]['static']:>7.4f} | {r[s]['maml']-r[s]['static']:>+8.4f}")
    print("-"*60)
    print(f"mean MAML   AUROC : {m.mean():.4f}")
    print(f"mean Static AUROC : {st.mean():.4f}")
    print(f"mean difference   : {mean_d:+.4f}")
    print(f"95% CI of diff    : [{ci[0]:+.4f}, {ci[1]:+.4f}]")
    print(f"paired t-test p (one-sided, MAML>Static): {t_p:.4f}")
    print(f"Wilcoxon p (one-sided)                  : {w_p:.4f}")
    verdict = "CI excludes 0 -> edge holds" if ci[0]>0 else "CI includes 0 -> edge not established; reframe on near-skyline transfer"
    print("VERDICT:", verdict)

for tag in ["WADI_to_SWaT","SWaT_to_WADI"]:
    if RESULTS.get(tag): paired_report(tag)


DIRECTION: WADI_to_SWaT   (n=6 training seeds)
seed |    MAML |  Static |     diff
   0 |  0.7917 |  0.7962 |  -0.0045
   1 |  0.7820 |  0.8149 |  -0.0329
   2 |  0.8062 |  0.7959 |  +0.0103
   3 |  0.7981 |  0.8039 |  -0.0058
   4 |  0.7912 |  0.7999 |  -0.0087
   5 |  0.8082 |  0.7706 |  +0.0376
------------------------------------------------------------
mean MAML   AUROC : 0.7962
mean Static AUROC : 0.7969
mean difference   : -0.0007
95% CI of diff    : [-0.0252, +0.0239]
paired t-test p (one-sided, MAML>Static): 0.5265
Wilcoxon p (one-sided)                  : 0.5781
VERDICT: CI includes 0 -> edge not established; reframe on near-skyline transfer
DIRECTION: SWaT_to_WADI   (n=6 training seeds)
seed |    MAML |  Static |     diff
   0 |  0.5823 |  0.5883 |  -0.0060
   1 |  0.5467 |  0.5998 |  -0.0531
   2 |  0.6359 |  0.5966 |  +0.0393
   3 |  0.5473 |  0.5912 |  -0.0439
   4 |  0.5377 |  0.5945 |  -0.0568
   5 |  0.5955 |  0.6017 |  -0.0062
-----------------------------------------